# Glacial Lake Expansion & GLOF (Glacial Lake Outburst Flood) Vulnerability Forecasting
### Multi-Task Deep Learning Benchmark: Random Forest vs. Multi-Task LSTM vs. TCN vs. DLinear
**Study Region**: Ladakh, Zanskar, Karakoram, and Sikkim Himalaya (2015–2024)

**Research Synthesis**:
- **Remote Sensing Mapper**: *Chen et al. / HMA Glacial Lake Inventory* — Multi-temporal lake boundary extraction via Sentinel-2 (NDWI) and Sentinel-1 (SAR).
- **Forecasting Framework**: *IceWatch: A Multi-Modal Deep Learning System for Glacial Lake Outburst Flood Prediction (arXiv, 2024)*.

**Core Objectives**:
1. **Continuous Forecasting**: Predict future water surface area ($km^2$) for high-risk proglacial and moraine-dammed lakes.
2. **Critical Hazard Early-Warning**: Classify GLOF vulnerability tier (0 = Low, 1 = Moderate, 2 = High / Critical Alert).

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, accuracy_score, f1_score, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__}")
print(f"Compute Device:  {device}")
if torch.cuda.is_available():
    print(f"GPU Name:        {torch.cuda.get_device_name(0)}")

In [ ]:
# Mount Google Drive if in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

DRIVE_CSV = '/content/drive/MyDrive/glof_data/himalayan_glacial_lakes_real.csv'
LOCAL_CSV = 'data/himalayan_glacial_lakes_2015_2024.csv'

if os.path.exists(DRIVE_CSV):
    print(f"-> SUCCESS: Found REAL GEE Glacial Lake dataset in Drive: {DRIVE_CSV}")
    df = pd.read_csv(DRIVE_CSV)
elif os.path.exists(LOCAL_CSV):
    print(f"-> Loading pre-compiled HMA reference lake dataset: {LOCAL_CSV}")
    df = pd.read_csv(LOCAL_CSV)
else:
    print("-> Fetching dataset from GitHub/generator...")
    url = "https://raw.githubusercontent.com/HetharthSahu/Glacier_Segment_TimeSeries/main/glof_lake_forecasting/data/himalayan_glacial_lakes_2015_2024.csv"
    try:
        df = pd.read_csv(url)
    except Exception:
        from data.generate_glof_dataset import generate_glof_dataset
        df = generate_glof_dataset()

# Clean metadata columns if coming from GEE
for c in ['system:index', '.geo']:
    if c in df.columns:
        df = df.drop(columns=[c])

df = df.sort_values(['lake_id', 'year', 'month']).reset_index(drop=True)

# Enrich geomorphic and hazard tiers if loaded directly from satellite GEE export
if 'dam_freeboard_m' not in df.columns:
    print('-> Computing derived geomorphic freeboard and hazard tiers from satellite time series...')
    if 'region' not in df.columns:
        df['region'] = df['lake_name'].apply(lambda x: 'Ladakh' if 'Gya' in str(x) or 'Drang' in str(x) or 'Pensilungpa' in str(x) else ('Sikkim' if 'Lhonak' in str(x) else 'Karakoram/Himalaya'))
    freeboards, distances, risks = [], [], []
    for lid in df['lake_id'].unique():
        sub = df[df['lake_id'] == lid]
        base_a = sub['lake_area_km2'].iloc[0]
        for _, row in sub.iterrows():
            area_ratio = row['lake_area_km2'] / max(base_a, 0.05)
            fb = max(4.0, 38.0 - (area_ratio - 1.0) * 40.0)
            dist = max(30.0, 850.0 - (area_ratio - 1.0) * 600.0)
            if fb < 15.0 or dist < 100.0 or row['sar_backscatter_db'] < -21.0:
                tier = 2
            elif fb < 25.0:
                tier = 1
            else:
                tier = 0
            freeboards.append(fb)
            distances.append(dist)
            risks.append(tier)
    df['dam_freeboard_m'] = freeboards
    df['dist_to_glacier_ice_m'] = distances
    df['glof_risk_tier'] = risks

print(f"Dataset Loaded: {len(df)} monthly observations across {df['lake_id'].nunique()} high-risk lakes (2015–2024).")
df.head(6)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Area expansion trajectories of benchmark high-risk lakes
available_lakes = list(df['lake_id'].unique())
candidate_lakes = ['GL_01_Gya', 'GL_02_DrangDrung', 'GL_03_Pensilungpa', 'GL_11_Kyagar', 'GL_31_SouthLhonak', 'GL_03_Kyagar', 'GL_04_SouthLhonak']
sample_lakes = [lid for lid in candidate_lakes if lid in available_lakes][:4]
if len(sample_lakes) < 2:
    sample_lakes = available_lakes[:4]

for lid in sample_lakes:
    sub = df[df['lake_id'] == lid].sort_values(['year', 'month'])
    if len(sub) == 0:
        continue
    reg_str = sub['region'].iloc[0] if 'region' in sub.columns else 'HMA'
    axes[0].plot(range(len(sub)), sub['lake_area_km2'], label=f"{sub['lake_name'].iloc[0]} ({reg_str})", linewidth=2)

axes[0].set_title("Multi-Temporal Glacial Lake Area Expansion (2015–2024)", fontsize=13, weight='bold')
axes[0].set_xlabel("Observation Month (May–Oct)")
axes[0].set_ylabel("Water Surface Area (km²)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Dam Freeboard vs GLOF Risk Tier
sns.scatterplot(data=df, x='lake_area_km2', y='dam_freeboard_m', hue='glof_risk_tier', palette='coolwarm', ax=axes[1], alpha=0.7)
axes[1].axhline(15.0, color='red', linestyle='--', label='Critical Freeboard Threshold (15m)')
axes[1].set_title("Dam Freeboard vs Lake Area (GLOF Alert Tiers)", fontsize=13, weight='bold')
axes[1].set_xlabel("Lake Surface Area (km²)")
axes[1].set_ylabel("Moraine Dam Freeboard (m)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
feature_cols = ['lake_area_km2', 'ndwi_mean', 'sar_backscatter_db', 'temp_summer_c', 'rainfall_peak_mm', 'dam_freeboard_m', 'dist_to_glacier_ice_m']
LOOKBACK = 6 # 6 observation months history
HORIZON = 1

train_X, train_y_reg, train_y_cls = [], [], []
val_X, val_y_reg, val_y_cls = [], [], []
test_X, test_y_reg, test_y_cls = [], [], []

for lid in df['lake_id'].unique():
    ldf = df[df['lake_id'] == lid].sort_values(['year', 'month']).reset_index(drop=True)
    years = ldf['year'].values
    mat = ldf[feature_cols].values
    target_area = ldf['lake_area_km2'].values
    target_risk = ldf['glof_risk_tier'].values

    for i in range(len(years) - LOOKBACK - HORIZON + 1):
        wx = mat[i : i + LOOKBACK]
        wy_reg = target_area[i + LOOKBACK + HORIZON - 1]
        wy_cls = target_risk[i + LOOKBACK + HORIZON - 1]
        pyr = years[i + LOOKBACK + HORIZON - 1]

        if pyr <= 2021:
            train_X.append(wx); train_y_reg.append(wy_reg); train_y_cls.append(wy_cls)
        elif pyr <= 2023:
            val_X.append(wx); val_y_reg.append(wy_reg); val_y_cls.append(wy_cls)
        else:
            test_X.append(wx); test_y_reg.append(wy_reg); test_y_cls.append(wy_cls)

train_X = np.array(train_X, dtype=np.float32)
train_y_reg = np.array(train_y_reg, dtype=np.float32).reshape(-1, 1)
train_y_cls = np.array(train_y_cls, dtype=np.int64)

val_X, val_y_reg, val_y_cls = np.array(val_X, dtype=np.float32), np.array(val_y_reg, dtype=np.float32).reshape(-1, 1), np.array(val_y_cls, dtype=np.int64)
test_X, test_y_reg, test_y_cls = np.array(test_X, dtype=np.float32), np.array(test_y_reg, dtype=np.float32).reshape(-1, 1), np.array(test_y_cls, dtype=np.int64)

# Feature normalization strictly on train partition
scaler = StandardScaler().fit(train_X.reshape(-1, len(feature_cols)))
train_X_s = scaler.transform(train_X.reshape(-1, len(feature_cols))).reshape(train_X.shape)
val_X_s = scaler.transform(val_X.reshape(-1, len(feature_cols))).reshape(val_X.shape)
test_X_s = scaler.transform(test_X.reshape(-1, len(feature_cols))).reshape(test_X.shape)

target_scaler = StandardScaler().fit(train_y_reg)
train_y_s = target_scaler.transform(train_y_reg)
val_y_s = target_scaler.transform(val_y_reg)
test_y_s = target_scaler.transform(test_y_reg)

class SeqDataset(Dataset):
    def __init__(self, X, y_reg, y_cls):
        self.X, self.y_reg, self.y_cls = torch.from_numpy(X).float(), torch.from_numpy(y_reg).float(), torch.from_numpy(y_cls).long()
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y_reg[idx], self.y_cls[idx]

train_loader = DataLoader(SeqDataset(train_X_s, train_y_s, train_y_cls), batch_size=32, shuffle=True)
val_loader = DataLoader(SeqDataset(val_X_s, val_y_s, val_y_cls), batch_size=32, shuffle=False)
test_loader = DataLoader(SeqDataset(test_X_s, test_y_s, test_y_cls), batch_size=32, shuffle=False)

print(f"Chronological Split: {len(train_X)} train (<=2021), {len(val_X)} val (2022-2023), {len(test_X)} test (2024).")

In [ ]:
# Model 2: Multi-Task LSTM
class GLOFLSTM(nn.Module):
    def __init__(self, in_dim=7, hidden=64, num_layers=2, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, num_layers=num_layers, batch_first=True, dropout=0.2)
        self.area_head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))
        self.risk_head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, num_classes))
    def forward(self, x):
        out, _ = self.lstm(x)
        feat = out[:, -1, :]
        return self.area_head(feat), self.risk_head(feat)

# Model 3: Multi-Task TCN
class Chomp1d(nn.Module):
    def __init__(self, s): super().__init__(); self.s = s
    def forward(self, x): return x[:, :, :-self.s].contiguous() if self.s > 0 else x

class TemporalBlock(nn.Module):
    def __init__(self, n_in, n_out, k=2, d=1):
        super().__init__()
        pad = (k - 1) * d
        self.net = nn.Sequential(
            nn.Conv1d(n_in, n_out, k, padding=pad, dilation=d), Chomp1d(pad), nn.ReLU(), nn.Dropout(0.2),
            nn.Conv1d(n_out, n_out, k, padding=pad, dilation=d), Chomp1d(pad), nn.ReLU(), nn.Dropout(0.2)
        )
        self.down = nn.Conv1d(n_in, n_out, 1) if n_in != n_out else None
        self.relu = nn.ReLU()
    def forward(self, x):
        res = x if self.down is None else self.down(x)
        return self.relu(self.net(x) + res)

class GLOFTCN(nn.Module):
    def __init__(self, in_dim=7, channels=[32, 64], k=2, num_classes=3):
        super().__init__()
        layers = []
        for i, ch in enumerate(channels):
            layers.append(TemporalBlock(in_dim if i==0 else channels[i-1], ch, k=k, d=2**i))
        self.net = nn.Sequential(*layers)
        self.area_head = nn.Linear(channels[-1], 1)
        self.risk_head = nn.Linear(channels[-1], num_classes)
    def forward(self, x):
        y = self.net(x.transpose(1, 2))
        feat = y[:, :, -1]
        return self.area_head(feat), self.risk_head(feat)

# Model 4: Multi-Task DLinear
class GLOFDLinear(nn.Module):
    def __init__(self, lookback=6, in_dim=7, num_classes=3, k=3):
        super().__init__()
        self.k = k
        self.avg = nn.AvgPool1d(kernel_size=k, stride=1, padding=0)
        in_flat = lookback * in_dim
        self.l_trend = nn.Linear(in_flat, 1)
        self.l_seasonal = nn.Linear(in_flat, 1)
        self.risk_head = nn.Linear(in_flat, num_classes)
    def forward(self, x):
        xt = x.transpose(1, 2)
        pl, pr = (self.k - 1) // 2, (self.k - 1) - (self.k - 1) // 2
        xp = torch.cat([xt[:, :, 0:1].repeat(1, 1, pl), xt, xt[:, :, -1:].repeat(1, 1, pr)], dim=-1)
        trend = self.avg(xp).transpose(1, 2)
        seasonal = x - trend
        s_flat, t_flat = seasonal.reshape(x.shape[0], -1), trend.reshape(x.shape[0], -1)
        return self.l_seasonal(s_flat) + self.l_trend(t_flat), self.risk_head(t_flat)

print("Multi-Task Deep Learning Models defined successfully!")

In [ ]:
def train_glof_pt(model, epochs=35, lr=0.002):
    model.to(device)
    crit_reg = nn.MSELoss()
    crit_cls = nn.CrossEntropyLoss()
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best_loss, best_w = float('inf'), None

    for ep in range(epochs):
        model.train()
        for xb, yr, yc in train_loader:
            xb, yr, yc = xb.to(device), yr.to(device), yc.to(device)
            opt.zero_grad()
            pa, pr = model(xb)
            loss = crit_reg(pa, yr) + 0.5 * crit_cls(pr, yc)
            loss.backward(); opt.step()

        model.eval()
        vloss = 0.0
        with torch.no_grad():
            for xb, yr, yc in val_loader:
                xb, yr, yc = xb.to(device), yr.to(device), yc.to(device)
                pa, pr = model(xb)
                l = crit_reg(pa, yr) + 0.5 * crit_cls(pr, yc)
                vloss += l.item() * len(xb)
        vloss /= len(val_loader.dataset)
        if vloss < best_loss:
            best_loss = vloss; best_w = model.state_dict().copy()
    model.load_state_dict(best_w)
    return model

y_test_unscaled = test_y_reg.ravel()
results = []

# 1. Random Forest Baseline
print("Training Random Forest Baseline...")
t0 = time.time()
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_cls = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
X_tr_flat = train_X_s.reshape(len(train_X_s), -1)
X_te_flat = test_X_s.reshape(len(test_X_s), -1)
rf_reg.fit(X_tr_flat, train_y_reg.ravel())
rf_cls.fit(X_tr_flat, train_y_cls)
rf_area = rf_reg.predict(X_te_flat)
rf_risk = rf_cls.predict(X_te_flat)
t_rf = round(time.time() - t0, 2)
results.append({"Model": "Random Forest", "Area_R2": r2_score(y_test_unscaled, rf_area), "RMSE_km2": np.sqrt(mean_squared_error(y_test_unscaled, rf_area)), "MAE_km2": mean_absolute_error(y_test_unscaled, rf_area), "Alert_Acc_pct": accuracy_score(test_y_cls, rf_risk)*100.0, "Risk_F1": f1_score(test_y_cls, rf_risk, average='macro'), "Time_s": t_rf})

# 2. LSTM
print("Training Multi-Task LSTM...")
t0 = time.time(); lstm = train_glof_pt(GLOFLSTM(in_dim=7, hidden=64))
lstm.eval()
with torch.no_grad():
    pa, pr = lstm(torch.from_numpy(test_X_s).to(device))
    lstm_area = target_scaler.inverse_transform(pa.cpu().numpy()).ravel()
    lstm_risk = torch.argmax(pr, dim=1).cpu().numpy()
t_lstm = round(time.time() - t0, 2)
results.append({"Model": "LSTM", "Area_R2": r2_score(y_test_unscaled, lstm_area), "RMSE_km2": np.sqrt(mean_squared_error(y_test_unscaled, lstm_area)), "MAE_km2": mean_absolute_error(y_test_unscaled, lstm_area), "Alert_Acc_pct": accuracy_score(test_y_cls, lstm_risk)*100.0, "Risk_F1": f1_score(test_y_cls, lstm_risk, average='macro'), "Time_s": t_lstm})

# 3. TCN
print("Training Multi-Task TCN...")
t0 = time.time(); tcn = train_glof_pt(GLOFTCN(in_dim=7))
tcn.eval()
with torch.no_grad():
    pa, pr = tcn(torch.from_numpy(test_X_s).to(device))
    tcn_area = target_scaler.inverse_transform(pa.cpu().numpy()).ravel()
    tcn_risk = torch.argmax(pr, dim=1).cpu().numpy()
t_tcn = round(time.time() - t0, 2)
results.append({"Model": "TCN", "Area_R2": r2_score(y_test_unscaled, tcn_area), "RMSE_km2": np.sqrt(mean_squared_error(y_test_unscaled, tcn_area)), "MAE_km2": mean_absolute_error(y_test_unscaled, tcn_area), "Alert_Acc_pct": accuracy_score(test_y_cls, tcn_risk)*100.0, "Risk_F1": f1_score(test_y_cls, tcn_risk, average='macro'), "Time_s": t_tcn})

# 4. DLinear
print("Training Multi-Task DLinear...")
t0 = time.time(); dlinear = train_glof_pt(GLOFDLinear(lookback=6, in_dim=7))
dlinear.eval()
with torch.no_grad():
    pa, pr = dlinear(torch.from_numpy(test_X_s).to(device))
    dlinear_area = target_scaler.inverse_transform(pa.cpu().numpy()).ravel()
    dlinear_risk = torch.argmax(pr, dim=1).cpu().numpy()
t_dlin = round(time.time() - t0, 2)
results.append({"Model": "DLinear", "Area_R2": r2_score(y_test_unscaled, dlinear_area), "RMSE_km2": np.sqrt(mean_squared_error(y_test_unscaled, dlinear_area)), "MAE_km2": mean_absolute_error(y_test_unscaled, dlinear_area), "Alert_Acc_pct": accuracy_score(test_y_cls, dlinear_risk)*100.0, "Risk_F1": f1_score(test_y_cls, dlinear_risk, average='macro'), "Time_s": t_dlin})

res_df = pd.DataFrame(results).round(4)
res_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Area Forecasting R2 and Alert Accuracy Comparison
sns.barplot(data=res_df, x='Model', y='Area_R2', hue='Model', ax=axes[0], palette='Reds_d', legend=False)
axes[0].set_title("Lake Area Forecast Accuracy (R² Score)", fontsize=13, weight='bold')
axes[0].set_ylim(min(0.85, res_df['Area_R2'].min() - 0.05), 1.0)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.4f}", (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=10)

# Plot 2: Confusion Matrix for GLOF Critical Alert Detection (LSTM)
cm = confusion_matrix(test_y_cls, lstm_risk)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1], xticklabels=['Low', 'Moderate', 'Critical Alert'], yticklabels=['Low', 'Moderate', 'Critical Alert'])
axes[1].set_title("GLOF Hazard Classification Confusion Matrix (LSTM)", fontsize=13, weight='bold')
axes[1].set_xlabel("Predicted Hazard Tier")
axes[1].set_ylabel("Actual Ground Truth Hazard")

plt.tight_layout()
plt.show()

## Automated Export & Download: Save All Results, Metrics, Figures & Model Checkpoints
This cell saves:
1. **`comparative_glof_metrics.csv`**: Full model evaluation table (Area $R^2$, RMSE, MAE, Alert Acc %, Macro F1, Training time).
2. **`glof_test_predictions_2024.csv`**: Ground truth vs predictions across all 4 models on the 2024 test horizon.
3. **`glof_model_benchmark_comparison.png`** & **`glof_actual_vs_predicted_series.png`**: Publication-quality 300 DPI figures.
4. **`best_glof_lstm.pt`**, **`best_glof_tcn.pt`**, **`best_glof_dlinear.pt`**: Trained PyTorch model weights.
5. **`glof_results_bundle.zip`**: Automatically copied to Google Drive (`MyDrive/glof_data/results/`) and downloaded to your computer.

In [ ]:
# ==============================================================================
# SAVE ALL RESULTS, METRICS, PREDICTIONS, FIGURES & MODEL WEIGHTS
# ==============================================================================
import os
import shutil
import zipfile

# Determine output directories (both Google Drive and local)
output_dirs = ['./glof_results']
drive_results_dir = '/content/drive/MyDrive/glof_data/results'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(drive_results_dir, exist_ok=True)
    output_dirs.append(drive_results_dir)

for d in output_dirs:
    os.makedirs(d, exist_ok=True)

print(f"-> Exporting results to: {output_dirs}")

# 1. Save Benchmark Metrics Summary CSV
for d in output_dirs:
    res_df.to_csv(os.path.join(d, 'comparative_glof_metrics.csv'), index=False)
print("  [OK] Saved comparative_glof_metrics.csv")

# 2. Save 2024 Test Set Predictions Table (Actual vs All 4 Models)
preds_df = pd.DataFrame({
    'Actual_Area_km2': y_test_unscaled,
    'RF_Area_km2': rf_area,
    'LSTM_Area_km2': lstm_area,
    'TCN_Area_km2': tcn_area,
    'DLinear_Area_km2': dlinear_area,
    'Actual_GLOF_Risk_Tier': test_y_cls,
    'RF_Risk_Tier': rf_risk,
    'LSTM_Risk_Tier': lstm_risk,
    'TCN_Risk_Tier': tcn_risk,
    'DLinear_Risk_Tier': dlinear_risk
})
for d in output_dirs:
    preds_df.to_csv(os.path.join(d, 'glof_test_predictions_2024.csv'), index=False)
print("  [OK] Saved glof_test_predictions_2024.csv")

# 3. Save PyTorch Model Weights (.pt files)
for d in output_dirs:
    torch.save(lstm.state_dict(), os.path.join(d, 'best_glof_lstm.pt'))
    torch.save(tcn.state_dict(), os.path.join(d, 'best_glof_tcn.pt'))
    torch.save(dlinear.state_dict(), os.path.join(d, 'best_glof_dlinear.pt'))
print("  [OK] Saved PyTorch model weights (LSTM, TCN, DLinear)")

# 4. Save High-Resolution Benchmark Figures (300 DPI)
fig_bench, axes_b = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=res_df, x='Model', y='Area_R2', hue='Model', ax=axes_b[0], palette='Reds_d', legend=False)
axes_b[0].set_title("Lake Area Forecast Accuracy (R² Score)", fontsize=13, weight='bold')
axes_b[0].set_ylim(min(0.85, res_df['Area_R2'].min() - 0.05), 1.0)
for p in axes_b[0].patches:
    axes_b[0].annotate(f"{p.get_height():.4f}", (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=10)

cm = confusion_matrix(test_y_cls, lstm_risk)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', ax=axes_b[1], xticklabels=['Low', 'Moderate', 'Critical Alert'], yticklabels=['Low', 'Moderate', 'Critical Alert'])
axes_b[1].set_title("GLOF Hazard Classification Confusion Matrix (LSTM)", fontsize=13, weight='bold')
axes_b[1].set_xlabel("Predicted Hazard Tier")
axes_b[1].set_ylabel("Actual Ground Truth Hazard")
plt.tight_layout()

for d in output_dirs:
    fig_bench.savefig(os.path.join(d, 'glof_model_benchmark_comparison.png'), dpi=300, bbox_inches='tight')
plt.close(fig_bench)
print("  [OK] Saved glof_model_benchmark_comparison.png (300 DPI)")

# 5. Save Actual vs Predicted Time-Series Horizon Plot
fig_ts, ax_ts = plt.subplots(figsize=(14, 5))
sample_range = range(min(60, len(y_test_unscaled)))
ax_ts.plot(sample_range, y_test_unscaled[:60], label='Actual Satellite Lake Area (km²)', color='black', linewidth=2.5, marker='o', markersize=4)
ax_ts.plot(sample_range, lstm_area[:60], label='LSTM Forecast', linestyle='--', linewidth=1.8, color='crimson')
ax_ts.plot(sample_range, tcn_area[:60], label='TCN Forecast', linestyle=':', linewidth=1.8, color='dodgerblue')
ax_ts.plot(sample_range, dlinear_area[:60], label='DLinear Forecast', linestyle='-.', linewidth=1.8, color='green')
ax_ts.set_title("GLOF Test Horizon: Actual Satellite Lake Area vs Deep Learning Forecasts (2024)", fontsize=13, weight='bold')
ax_ts.set_xlabel("Test Horizon Steps (2024 Ice-Free Months)")
ax_ts.set_ylabel("Lake Surface Area (km²)")
ax_ts.legend(loc='upper right')
ax_ts.grid(True, alpha=0.3)
plt.tight_layout()

for d in output_dirs:
    fig_ts.savefig(os.path.join(d, 'glof_actual_vs_predicted_series.png'), dpi=300, bbox_inches='tight')
plt.close(fig_ts)
print("  [OK] Saved glof_actual_vs_predicted_series.png (300 DPI)")

# 6. Bundle Everything into a Single Zip File for Easy 1-Click Download
zip_path = 'glof_results_bundle.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk('./glof_results'):
        for file in files:
            full_p = os.path.join(root, file)
            arcname = os.path.relpath(full_p, './glof_results')
            zipf.write(full_p, arcname)

if os.path.exists(drive_results_dir):
    shutil.copy(zip_path, os.path.join(drive_results_dir, 'glof_results_bundle.zip'))
    print(f"\n-> SUCCESS: Complete results bundle copied to Google Drive: {drive_results_dir}/glof_results_bundle.zip")

print(f"-> Local ZIP bundle ready: {os.path.abspath(zip_path)} ({round(os.path.getsize(zip_path)/1024, 1)} KB)")

# Trigger browser download if in Colab
try:
    from google.colab import files
    print("\nInitiating direct browser download of glof_results_bundle.zip...")
    files.download(zip_path)
except Exception:
    print("\nTo download manually in Colab, run: files.download('glof_results_bundle.zip')")